In [29]:
import pandas as pd
import numpy as np
import warnings 
import holidays
import matplotlib.pyplot as plt
import koreanize_matplotlib
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
warnings.filterwarnings('ignore')

In [30]:
df2024 = pd.read_csv('../../Data/Zero/2024_data.csv')
df2025 = pd.read_csv('../../Data/Zero/2025_data.csv')

df = pd.concat([df2024,df2025],axis=0)

In [31]:
# 1. 시작 대여소 기준으로만 필터링 (순수 대여량 예측 목적)
target_station = ['ST-1035', 'ST-454', 'ST-471']
df['기준_날짜'] = pd.to_datetime(df['기준_날짜'])
df['year'] = df['기준_날짜'].dt.year
df['weekday'] = df['기준_날짜'].dt.dayofweek
df['day_type'] = np.where(df['weekday'] < 5, 0, 1)
df = df[(df['전체_이용_분'] >= 5) & (df['전체_이용_거리'] >= 5)]

df['datetime'] = pd.to_datetime(df['기준_날짜']) + pd.to_timedelta(df['시간대'], unit='h')
df = df[df['시작_대여소_ID'].isin(target_station)].copy()
df_hourly = df.groupby(['시작_대여소_ID', 'datetime','year']).agg({
    '전체_건수': 'sum',
    '온도': 'mean',
    '습도': 'mean',
    '불쾌지수': 'mean',
    '강수량': 'mean',
    '적설량': 'mean'
}).reset_index()


In [32]:
# 1. 시간대와 요일 정보 다시 추출
df_hourly['hour'] = df_hourly['datetime'].dt.hour
df_hourly['weekday'] = df_hourly['datetime'].dt.dayofweek
df_hourly['is_weekend'] = np.where(df_hourly['weekday'] < 5, 0, 1)

# 2. 대여소 ID를 숫자로 변환 (HGB가 대여소별 특징을 알게 함)
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
df_hourly['station_idx'] = le.fit_transform(df_hourly['시작_대여소_ID'])
# 반드시 대여소별로 정렬 후 계산해야 데이터가 꼬이지 않습니다.
df_hourly = df_hourly.sort_values(['시작_대여소_ID', 'datetime'])

# 1. 과거 데이터 추가 (Lag Features)
# 한 시간 전, 두 시간 전 대여량
df_hourly['lag_1h'] = df_hourly.groupby('시작_대여소_ID')['전체_건수'].shift(1)
df_hourly['lag_2h'] = df_hourly.groupby('시작_대여소_ID')['전체_건수'].shift(2)

# 2. 어제 같은 시간 대여량 (따릉이는 패턴이 반복됩니다)
df_hourly['lag_24h'] = df_hourly.groupby('시작_대여소_ID')['전체_건수'].shift(24)

# 3. 최근 3시간 평균 대여량 (최근의 추세 반영)
df_hourly['rolling_3h'] = df_hourly.groupby('시작_대여소_ID')['전체_건수'].transform(lambda x: x.shift(1).rolling(3).mean())

# 결측치 제거 (과거 데이터가 없는 초기 행들)
df_hourly = df_hourly.dropna()

# 피처 리스트에 추가
# 3. 피처 리스트 대폭 수정
features = [
    'station_idx', # 어느 대여소인지
    'hour',        # 몇 시인지 (가장 중요!)
    'weekday',     # 무슨 요일인지
    'is_weekend',  # 평일/주말
    '온도', '습도', '불쾌지수', '강수량', '적설량',
    'lag_1h', 'lag_2h', 'lag_24h', 'rolling_3h'
]

# 4. (선택사항) 타겟값 로그 변환
# 대여량은 편차가 크므로 로그를 취하면 학습이 더 잘됩니다.
# y_train = np.log1p(train[target])

In [33]:
df_hourly

,시작_대여소_ID,datetime,year,전체_건수,온도,습도,불쾌지수,강수량,적설량,hour,weekday,is_weekend,station_idx,lag_1h,lag_2h,lag_24h,rolling_3h
24,ST-1035,2024-01-02 16:00:00,2024,2.0,3.7,59.0,43.02117,0.0,0.0,16,1,0,0,2.0,2.0,2.0,2.000000
25,ST-1035,2024-01-02 17:00:00,2024,4.0,3.0,65.0,41.36550,0.0,0.0,17,1,0,0,2.0,2.0,2.0,2.000000
26,ST-1035,2024-01-02 18:00:00,2024,4.0,1.4,76.0,37.61936,0.0,0.0,18,1,0,0,4.0,2.0,2.0,2.666667
27,ST-1035,2024-01-02 19:00:00,2024,8.0,0.9,79.0,36.43589,0.0,0.0,19,1,0,0,4.0,4.0,2.0,3.333333
28,ST-1035,2024-01-02 20:00:00,2024,4.0,-0.1,86.0,33.83586,0.0,0.0,20,1,0,0,8.0,4.0,2.0,5.333333
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
31896,ST-471,2025-12-31 14:00:00,2025,2.0,-4.3,30.0,37.24990,0.0,0.0,14,2,0,2,2.0,2.0,4.0,2.333333
31897,ST-471,2025-12-31 15:00:00,2025,4.0,-4.4,31.0,36.95264,0.0,0.0,15,2,0,2,2.0,2.0,2.0,2.000000
31898,ST-471,2025-12-31 18:00:00,2025,1.0,-6.6,42.0,32.20372,0.0,0.0,18,2,0,2,4.0,2.0,2.0,2.666667
31899,ST-471,2025-12-31 19:00:00,2025,1.0,-7.2,43.0,31.25396,0.0,0.0,19,2,0,2,1.0,4.0,2.0,2.333333


In [34]:
target = '전체_건수'
# features = [
#     '온도',
#     '습도',
#     '불쾌지수',
#     '강수량',
#     '적설량',
# ]

In [35]:
train = df_hourly[df_hourly['year'] == 2024]
test  = df_hourly[df_hourly['year'] == 2025]

X_train = train[features]
y_train = train[target]

X_test = test[features]
y_test = test[target]

from sklearn.ensemble import HistGradientBoostingRegressor

hgb = HistGradientBoostingRegressor(
    learning_rate=0.03,
    max_iter=600,
    max_depth=10,
    min_samples_leaf=10,
    random_state=42
)

hgb.fit(X_train, y_train)
pred_hgb = hgb.predict(X_test)

print("HGB MAE:", mean_absolute_error(y_test, pred_hgb))
print("HGB RMSE:", np.sqrt(mean_squared_error(y_test, pred_hgb)))
print("HGB R2:", r2_score(y_test, pred_hgb))

HGB MAE: 2.368251052333995
HGB RMSE: 3.287496268710379
HGB R2: 0.5105151758791893


In [36]:
import lightgbm as lgb

lgb_model = lgb.LGBMRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=8,
    num_leaves=31,
    random_state=42
)

lgb_model.fit(X_train, y_train)
pred_lgb = lgb_model.predict(X_test)

print("LGB MAE:", mean_absolute_error(y_test, pred_lgb))
print("LGB RMSE:", np.sqrt(mean_squared_error(y_test, pred_lgb)))
print("LGB R2:", r2_score(y_test, pred_lgb))

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001240 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 909
[LightGBM] [Info] Number of data points in the train set: 16594, number of used features: 13
[LightGBM] [Info] Start training from score 6.087622
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: 